In [0]:
from pyspark.sql import functions as F

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, max as _max

# ------------------------------
# 1️ Define tables and consumer
# ------------------------------
SOURCE_TABLE = "fq_dev_catalog.silver.sales_bills"
SILVER_TABLE = "fq_dev_catalog.gold.sales_daily_performance"
VERSION_TABLE = "fq_dev_catalog.bronze.version_control"
CONSUMER_NAME = "gold_sales_new"                          # unique per notebook/consumer

# ------------------------------
# 2️ Get latest version from source Delta table
# ------------------------------
delta_table = DeltaTable.forName(spark, SOURCE_TABLE)

latest_version = (
    delta_table.history()
    .select(_max("version").alias("latest_version"))
    .collect()[0]["latest_version"]
)

# ------------------------------
# 3️ Get last processed version from VERSION_TABLE
# ------------------------------
if spark.catalog.tableExists(VERSION_TABLE):
    last_processed_version = (
        spark.table(VERSION_TABLE)
        .filter(col("source_table") == SOURCE_TABLE)
        .filter(col("consumer_name") == CONSUMER_NAME)
        .agg(_max("last_processed_version").alias("last_version"))
        .collect()[0]["last_version"]
    )
else:
    last_processed_version = None

last_processed_version = last_processed_version if last_processed_version is not None else -1

# ------------------------------
# 4️ Determine start_version safely
# ------------------------------
if last_processed_version >= latest_version:
    print(f"No new versions to process. Latest: {latest_version}, Last processed: {last_processed_version}")
    start_version = None
else:
    start_version = last_processed_version + 1
    print(f"Processing versions from {start_version} to {latest_version}")

In [0]:
from pyspark.sql.functions import col
 
SOURCE_TABLE = "fq_dev_catalog.silver.sales_bills"
 
# start_version and latest_version are assumed to be computed in previous cell
# start_version = last_processed_version + 1

# Check if there are new versions to process
if start_version is None:
    # No new data to process - create empty DataFrame with same schema as source table
    incremental_df = spark.table(SOURCE_TABLE).limit(0)
    print("No new versions to process. Using empty DataFrame.")
else:
    # 1️ Read incremental data using CDF
    incremental_df = (
        spark.read
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_version)
        .option("endingVersion", latest_version)
        .table(SOURCE_TABLE)
    )
    
    # 2️ Optional: keep only INSERT records
    incremental_df = incremental_df.filter(col("_change_type").isin("insert", "update_postimage"))
 
display(incremental_df)

In [0]:
affected_stores_df = (
    incremental_df
    .select("deployment_name")
    .distinct()
)

In [0]:
cutoff_date = (
    incremental_df
    .selectExpr("min(business_date) as min_date")
    .collect()[0]["min_date"]
)

In [0]:
recalc_df = (
    spark.read.table("fq_dev_catalog.silver.sales_bills")
    .join(affected_stores_df, "deployment_name", "inner")
    .filter(col("business_date") >= cutoff_date)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sales_base = (
    # spark.table("fq_dev_catalog.silver.sales_bills")
    recalc_df
    .groupBy(
        F.col("business_date"),
        F.col("deployment_name"),
        F.col("company_id"),
        F.col("country_code")
    )
    .agg(
        (
            F.sum(F.coalesce(F.col("bill_discount"), F.lit(0))) +
            F.sum(F.coalesce(F.col("total_items_discount"), F.lit(0)))
        ).alias("total_discount"),
        F.sum("net_amount_before_discount").alias("net_sales_before_discount"),
        F.sum("net_amount_after_discount").alias("net_sales_after_discount"),
        F.sum("gross_amount").alias("gross_sales"),
        F.sum("tax_amount").alias("tax_amount"),
        F.count("ng_id").alias("total_transactions"),
        F.sum("total_items").alias("total_items"),
        F.sum(F.when(F.col("is_void") == True, 1).otherwise(0)).alias("total_void")
    )
)

In [0]:
dim_date = spark.table("fq_dev_catalog.silver.dim_date")

sales_with_date = (
    sales_base.alias("s")
    .join(
        dim_date.alias("d"),
        F.col("s.business_date") == F.col("d.calendar_date"),
        "left"
    )
    .select(
        F.col("s.deployment_name").alias("store_name"),
        F.col("d.date_key"),
        F.col("s.business_date").alias("calendar_date"),
        F.col("s.company_id"),
        F.col("s.country_code"),
        F.col("s.total_discount"),
        F.col("s.net_sales_before_discount"),
        F.col("s.net_sales_after_discount"),
        F.col("s.gross_sales"),
        F.col("s.tax_amount"),
        F.col("s.total_transactions"),
        F.col("s.total_items"),
        F.col("s.total_void")
    )
)

In [0]:
dim_company = spark.table("fq_dev_catalog.silver.dim_company")

sales_with_company = (
    sales_with_date.alias("s")
    .join(
        dim_company.alias("c"),
        "company_id",
        "left"
    )
    .select(
        "s.*",
        F.col("c.company_name")
    )
)

In [0]:
dim_store = spark.table("fq_dev_catalog.silver.dim_store")

sales_with_store = (
    sales_with_company.alias("s")
    .join(
        dim_store.select("deployment_name", "zone").alias("st"),
        F.col("s.store_name") == F.col("st.deployment_name"),
        "left"
    )
    .withColumnRenamed("zone", "city_name")
)


In [0]:
budget = spark.table("fq_dev_catalog.silver.budget")

sales_with_budget = (
    sales_with_store.alias("s")
    .join(
        budget.alias("b"),
        (F.col("s.store_name") == F.col("b.deployment_name")) &
        (F.col("s.calendar_date") == F.col("b.effective_date")),
        "left"
    )
    .withColumn("budget", F.col("b.budget_amount"))
)


In [0]:
sales_with_budget=sales_with_budget.drop("budget","effective_date","budget_remarks","deployment_name","source_system")

In [0]:
sales_with_budget=sales_with_budget.withColumnRenamed("store_name", "deployment_name")

In [0]:
sales_metrics = sales_with_budget.withColumn(
    "avg_bill_value",
    F.when(
        F.col("total_transactions") > 0,
        F.col("net_sales_before_discount") / F.col("total_transactions")
    ).otherwise(F.lit(0))
)

In [0]:
mtd_window = (
    Window.partitionBy(
        "deployment_name",
        F.year("calendar_date"),
        F.month("calendar_date")
    )
    .orderBy("calendar_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

ytd_window = (
    Window.partitionBy(
        "deployment_name",
        F.year("calendar_date")
    )
    .orderBy("calendar_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

In [0]:
sales_metrics = (
    sales_metrics
    .withColumn("mtd_net_sales", F.sum("net_sales_before_discount").over(mtd_window))
    .withColumn("ytd_net_sales", F.sum("net_sales_before_discount").over(ytd_window))
)

In [0]:
import pyspark.sql.functions as F

ly_df = (
    sales_metrics
    .select(
        "deployment_name",
        F.add_months("calendar_date", 12).alias("calendar_date"),
        F.col("gross_sales").alias("ly_gross_sales")
    )
)



final_df = (
    sales_metrics.alias("c")
    .join(
        ly_df.alias("l"),
        ["deployment_name", "calendar_date"],
        "left"
    )
)


In [0]:
dim_brand = spark.table("fq_dev_catalog.silver.dim_brand")

final_df = (
    final_df.alias("f")
    .join(
        dim_brand.select("company_id", "brand_name").alias("b"),
        "company_id",
        "left"
    )
    .select("f.*", "b.brand_name")
)


In [0]:
# df_wastage=spark.table("fq_dev_catalog.silver.wastage")
# df_wastage.display()

In [0]:
df_cost= spark.read.table("fq_dev_catalog.silver.cost")

In [0]:
df_item_sales = (
    spark.read.table("fq_dev_catalog.silver.sales_items_silver_2")
    .join(affected_stores_df, "deployment_name", "inner")
    .filter(col("business_date") >= cutoff_date)
)

In [0]:
from pyspark.sql.functions import col, sum as _sum

daily_base_sales_df = (
    df_item_sales
    .filter(col("is_base_item") == True)
    .groupBy(
        col("business_date"),
        col("deployment_name"),
        col("item_upccode")
    )
    .agg(
        _sum("quantity").alias("daily_quantity"),
        _sum("net_item_amount").alias("daily_net_sales"),
        _sum("gross_item_amount").alias("daily_gross_sales")
    )
)

In [0]:
from pyspark.sql.functions import col

daily_base_cost_df = (
    daily_base_sales_df.alias("s")
    .join(
        df_cost.alias("c"),
        (
            (col("s.item_upccode") == col("c.upc_code")) &
            (col("s.deployment_name") == col("c.deployment_name")) &
            (col("s.business_date") >= col("c.effective_from_date")) &
            (
                col("c.effective_to_date").isNull() |
                (col("s.business_date") <= col("c.effective_to_date"))
            )
        ),
        "left"
    )
    .select(
        col("s.business_date"),
        col("s.deployment_name"),
        col("s.item_upccode"),
        col("s.daily_quantity"),
        col("c.unit_cost"),
        (col("s.daily_quantity") * col("c.unit_cost")).alias("daily_item_cost"),
        col("s.daily_net_sales"),
        col("s.daily_gross_sales")
    )
)


In [0]:
# df_cost.filter(col("deployment_name")=="ALBAIK - SQ DB01 - DUBAI MALL - 1005001").display()

In [0]:
from pyspark.sql.functions import col, sum as _sum

daily_base_cost_df = (
    daily_base_cost_df
    .groupBy(
        col("business_date"),
        col("deployment_name")
    )
    .agg(
        _sum("daily_item_cost").alias("total_base_item_cost")
    )
)

In [0]:
from pyspark.sql.functions import col, sum as _sum

daily_combo_child_sales_df = (
    df_item_sales
    .filter(col("is_combo_child_item") == True)
    .groupBy(
        col("business_date"),
        col("deployment_name"),
        col("item_upccode"),

    )
    .agg(
        _sum("quantity").alias("daily_quantity")
    )
)


In [0]:
daily_combo_child_cost_df = (
    daily_combo_child_sales_df.alias("s")
    .join(
        df_cost.alias("c"),
        (
            (col("s.item_upccode") == col("c.upc_code")) &
            (col("s.deployment_name") == col("c.deployment_name")) &
            (col("s.business_date") >= col("c.effective_from_date")) &
            (
                col("c.effective_to_date").isNull() |
                (col("s.business_date") <= col("c.effective_to_date"))
            )
        ),
        "left"
    )
    .select(
        col("s.business_date"),
        col("s.deployment_name"),
        (col("s.daily_quantity") * col("c.unit_cost")).alias("daily_item_cost")
    )
)


In [0]:
daily_store_combo_child_cost_df = (
    daily_combo_child_cost_df
    .groupBy("business_date", "deployment_name")
    .agg(
        _sum("daily_item_cost").alias("combo_child_item_cost")
    )
)


In [0]:
daily_addon_child_sales_df = (
    df_item_sales
    .filter(col("is_addon_child_item") == True)
    .groupBy(
        col("business_date"),
        col("deployment_name"),
        col("item_upccode")
     
    )
    .agg(
        _sum("quantity").alias("daily_quantity")
    )
)


In [0]:
daily_addon_child_cost_df = (
    daily_addon_child_sales_df.alias("s")
    .join(
        df_cost.alias("c"),
        (
            (col("s.item_upccode") == col("c.upc_code")) &
            (col("s.deployment_name") == col("c.deployment_name")) &
            (col("s.business_date") >= col("c.effective_from_date")) &
            (
                col("c.effective_to_date").isNull() |
                (col("s.business_date") <= col("c.effective_to_date"))
            )
        ),
        "left"
    )
    .select(
        col("s.business_date"),
        col("s.deployment_name"),
        (col("s.daily_quantity") * col("c.unit_cost")).alias("daily_item_cost")
    )
)


In [0]:
daily_store_addon_child_cost_df = (
    daily_addon_child_cost_df
    .groupBy("business_date", "deployment_name")
    .agg(
        _sum("daily_item_cost").alias("addon_child_item_cost")
    )
)

In [0]:
from pyspark.sql.functions import col, sum as _sum

daily_standalone_addon_sales_df = (
    df_item_sales
    .filter(
        (col("is_addon_item") == True) &
        (col("is_combo_item") == False)
    )
    .groupBy(
        col("business_date"),
        col("deployment_name"),
        col("item_upccode")
    )
    .agg(
        _sum("quantity").alias("daily_quantity")
    )
)

In [0]:
from pyspark.sql import functions as F

addon_parent_cost_df = (
    daily_standalone_addon_sales_df.alias("s")
    .join(
        df_cost.alias("c"),
        on=[
            F.col("s.item_upccode") == F.col("c.upc_code"),
            F.col("s.deployment_name") == F.col("c.deployment_name"),
            F.col("s.business_date").between(
                F.col("c.effective_from_date"),
                F.col("c.effective_to_date")
            )
        ],
        how="left"
    )
    .select(
        F.col("s.business_date"),
        F.col("s.deployment_name"),
        F.col("s.item_upccode"),
        F.col("s.daily_quantity"),
        F.coalesce(F.col("c.unit_cost"), F.lit(0.0)).alias("unit_cost")
    )
)


In [0]:
from pyspark.sql.functions import when

addon_parent_daily_cost_df = (
    addon_parent_cost_df
    .withColumn(
        "daily_cost",
        when(col("unit_cost").isNull(), 0)
        .otherwise(col("daily_quantity") * col("unit_cost"))
    )
)
final_addon_parent_cost_df = (
    addon_parent_daily_cost_df
    .groupBy("business_date", "deployment_name")
    .agg(
        _sum("daily_cost").alias("addon_parent_total_cost")
    )
)


In [0]:
from pyspark.sql.functions import col, coalesce, lit

final_daily_store_cost_df = (
    daily_base_cost_df.alias("base")
    .join(
        daily_store_combo_child_cost_df.alias("combo_child"),
        ["business_date", "deployment_name"],
        "left"
    )
    .join(
        daily_store_addon_child_cost_df.alias("addon_child"),
        ["business_date", "deployment_name"],
        "left"
    )
    .join(
        final_addon_parent_cost_df.alias("addon"),
        ["business_date", "deployment_name"],
        "left"
    )
    .select(
        col("business_date"),
        col("deployment_name"),

        coalesce(col("total_base_item_cost"), lit(0)).alias("total_base_item_cost"),
        coalesce(col("combo_child_item_cost"), lit(0)).alias("combo_child_item_cost"),
        coalesce(col("addon_child_item_cost"), lit(0)).alias("addon_child_item_cost"),
        coalesce(col("addon_parent_total_cost"), lit(0)).alias("addon_parent_total_cost"),

        (
            coalesce(col("total_base_item_cost"), lit(0)) +
            coalesce(col("combo_child_item_cost"), lit(0)) +
            coalesce(col("addon_child_item_cost"), lit(0)) +
            coalesce(col("addon_parent_total_cost"), lit(0))
        ).alias("total_cogs")
    )
)

In [0]:
# Remove duplicate columns by keeping only the first occurrence
# final_daily_store_cost_df.display()

In [0]:
from pyspark.sql import functions as F

final_df_with_cost = (
    final_df.alias("f")
    .join(
        final_daily_store_cost_df.alias("c"),
        on=[
            F.col("f.deployment_name") == F.col("c.deployment_name"),
            F.col("f.calendar_date") == F.col("c.business_date")
        ],
        how="left"
    )
    .select(
        "f.*",
        F.col("total_cogs").alias("daily_store_cost")
    )
)


In [0]:
# display(final_df_with_cost)

In [0]:
# final_df_with_cost.filter(col('calendar_date')=='2026-01-19').display()

In [0]:
final_df_with_margin = final_df_with_cost.withColumn("margin", col("net_sales_after_discount")-col("daily_store_cost"))
# final_df_with_margin.display()

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType

DECIMAL_18_3 = DecimalType(18, 3)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType, DoubleType, LongType, IntegerType

DECIMAL_18_3 = DecimalType(18, 3)

final_gold_df = (
    final_df_with_margin
        # ---- Financial Metrics ----
        .withColumn("total_discount", col("total_discount").cast(DECIMAL_18_3))
        .withColumn("net_sales_before_discount", col("net_sales_before_discount").cast(DECIMAL_18_3))
        .withColumn("net_sales_after_discount", col("net_sales_after_discount").cast(DECIMAL_18_3))
        .withColumn("gross_sales", col("gross_sales").cast(DECIMAL_18_3))
        .withColumn("tax_amount", col("tax_amount").cast(DECIMAL_18_3))
        .withColumn("avg_bill_value", col("avg_bill_value").cast(DECIMAL_18_3))
        .withColumn("mtd_net_sales", col("mtd_net_sales").cast(DECIMAL_18_3))
        .withColumn("ytd_net_sales", col("ytd_net_sales").cast(DECIMAL_18_3))
        .withColumn("ly_gross_sales", col("ly_gross_sales").cast(DECIMAL_18_3))

        # ---- Cost / Margin (convert DOUBLE → DECIMAL) ----
        .withColumn("budget_amount", col("budget_amount").cast(DECIMAL_18_3))
        .withColumn("daily_store_cost", col("daily_store_cost").cast(DECIMAL_18_3))
        .withColumn("margin", col("margin").cast(DECIMAL_18_3))
)

In [0]:
display(final_gold_df)

In [0]:
final_gold_df.count()

In [0]:
final_gold_df.createOrReplaceTempView("final_gold_df")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# ================================
# CONFIG
# ================================
GOLD_TABLE = "fq_dev_catalog.gold.sales_daily_performance"

# final_gold_df
#  ASSUMED: already recomputed from SILVER
#  ASSUMED: contains correct MTD / YTD values
#  ASSUMED: filtered to only impacted dates & deployments

# ================================
# STEP 1 — DELETE OLD GOLD RECORDS
# ================================
gold_target = DeltaTable.forName(spark, GOLD_TABLE)

gold_target.alias("tgt").delete(
    """
    EXISTS (
        SELECT 1
        FROM final_gold_df AS src
        WHERE
            tgt.deployment_name = src.deployment_name
            AND tgt.calendar_date = src.calendar_date
    )
    """
)

print("Old GOLD records deleted successfully")

# ================================
# STEP 2 — INSERT RECOMPUTED RECORDS
# ================================
(
    final_gold_df
    .write
    .mode("append")
    .saveAsTable(GOLD_TABLE)
)

print("New GOLD records inserted successfully")

# ================================
# OPTIONAL VALIDATION
# ================================
print(f"Inserted rows count : {final_gold_df.count()}")


In [0]:
# (
#     final_gold_df.write
#         .format("delta")
#         .mode("overwrite")
#         .option("overwriteSchema", "true")   # IMPORTANT
#         .saveAsTable("fq_dev_catalog.gold.sales_daily_performance")
# )

In [0]:
# %sql
# ALTER TABLE fq_dev_catalog.gold.sales_daily_performance
# SET TBLPROPERTIES (
#   'delta.enableChangeDataFeed' = 'true',
#   'delta.columnMapping.mode' = 'name'
# );


In [0]:

from pyspark.sql.functions import current_timestamp

SOURCE_TABLE = "fq_dev_catalog.silver.sales_bills"
CONSUMER_NAME = "gold_sales_new"
VERSION_TABLE = "fq_dev_catalog.bronze.version_control"

spark.sql(f"""
UPDATE {VERSION_TABLE}
SET
  last_processed_version = {latest_version},
  updated_at = current_timestamp()
WHERE
  source_table = '{SOURCE_TABLE}'
  AND consumer_name = '{CONSUMER_NAME}'
""")

In [0]:
%sql
SELECT * FROM fq_dev_catalog.bronze.version_control;

In [0]:
# df.count()